# Libraries and Data

In [1]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

df = pd.read_csv("../data/processed/ratings_clean.csv")
df.shape

(100836, 6)

# Collaborative Filtering

In [2]:
reader = Reader(rating_scale= (0.5, 5.0))
data = Dataset.load_from_df(df[["userId", "movieId", "rating"]], reader)

trainset, testset = train_test_split(data, test_size= 0.2, random_state= 42)

print(f"Train ratings: {trainset.n_ratings}")
print(f"Test ratings: {len(testset)}")

Train ratings: 80668
Test ratings: 20168


## Model training

In [3]:
model = SVD(n_factors= 100, random_state=42)
model.fit(trainset)

print("Model trained!")

Model trained!


In [4]:
from surprise import accuracy

predictions = model.test(testset)
print(f"RMSE: {accuracy.rmse(predictions):.4%}")
print(f"MAE: {accuracy.mae(predictions):.4%}")

RMSE: 0.8807
RMSE: 88.0746%
MAE:  0.6766
MAE: 67.6573%


### Predictions

In [6]:
movies = pd.read_csv('../data/raw/movies.csv')

In [10]:
def get_recommendations(user_id, n=10):
    # Get movies the user hasn't rated yet
    rated_movies = df[df["userId"] == user_id]["movieId"].tolist()
    all_movies = df["movieId"].unique()
    unrated = [m for m in all_movies if m not in rated_movies]

    # Predict ranges for all unrated movies
    predictions = [model.predict(user_id, movie_id) for movie_id in unrated]
    predictions.sort(key=lambda x: x.est, reverse= True)

    # Return Top n
    top_n = predictions[:n]
    movie_ids = [p.iid for p in top_n]
    scores = [round(p.est, 2) for p in top_n]

    result = movies[movies["movieId"].isin(movie_ids)].copy()
    result["predicted_rating"] = result["movieId"].map(dict(zip(movie_ids, scores)))
    result = result.sort_values("predicted_rating", ascending = False)
    result.insert(0, "rank", range(1, len(result) + 1))
    return result[['rank', 'title', 'genres']].reset_index(drop=True)


In [11]:
get_recommendations(user_id=1)

,rank,title,genres
0,1,Blade Runner (1982),Action|Sci-Fi|Thriller
1,2,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
2,3,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller
3,4,Casablanca (1942),Drama|Romance
4,5,One Flew Over the Cuckoo's Nest (1975),Drama
5,6,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi
6,7,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama
7,8,Lost in Translation (2003),Comedy|Drama|Romance
8,9,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
9,10,"Departed, The (2006)",Crime|Drama|Thriller


In [13]:
user_rated = (
    df[df['userId'] == 1]
    .sort_values('rating', ascending=False)
    [['title', 'genres', 'rating']]
)
print(user_rated.to_string(index=False))

                                                                         title                                                    genres  rating
                                                  M*A*S*H (a.k.a. MASH) (1970)                                          Comedy|Drama|War     5.0
                                                              Excalibur (1981)                                         Adventure|Fantasy     5.0
                                     Indiana Jones and the Last Crusade (1989)                                          Action|Adventure     5.0
                                                   Pink Floyd: The Wall (1982)                                             Drama|Musical     5.0
                                                  From Russia with Love (1963)                                 Action|Adventure|Thriller     5.0
                                                             Goldfinger (1964)                                 Action|Adventure|Th